# FL + Laplacian DP + Adaptive Quantization: full paper protocol on a Kaggle GPU

**Before running:** in the right panel set *Accelerator → GPU T4 x2* (or P100) and turn *Internet → On* (phone verification is needed once).

**To run without keeping the browser open:** click *Save Version → Save & Run All (Commit)*. The notebook then runs in the background; results appear under *Output* when it finishes.

Every run saves a checkpoint every evaluation, so if a session dies you can rerun the same cell and it resumes.

In [ ]:
# 1) get the code (replace YOUR_USERNAME with your GitHub user name)
!git clone https://github.com/YOUR_USERNAME/fl-dp-adaptive-quant.git
%cd fl-dp-adaptive-quant
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) unit tests (should print '10 passed')
!pip -q install pytest
!python -m pytest -q

In [ ]:
# 3) smoke test (~1 min)
!python scripts/run_suite.py --suite smoke --preset fast
!tail -n 3 results/logs/*smoke*.log

## Paper protocol (1000 rounds, 5 local epochs)
Rough GPU time per run: MNIST 15–30 min, CIFAR10 15–30 min. Running 2 jobs in parallel uses the GPU better. Run the suites you need; finished runs are skipped automatically.

In [ ]:
# Table 6 / Fig. 8: quantization schemes under DP (6 runs each)
!python scripts/run_suite.py --suite quant_mnist --preset paper --jobs 2
!python scripts/run_suite.py --suite quant_cifar10 --preset paper --jobs 2

In [ ]:
# Tables 3 and 5: clipping bound and privacy budget (MNIST)
!python scripts/run_suite.py --suite clip_mnist --preset paper --jobs 2
!python scripts/run_suite.py --suite dp_mnist --preset paper --jobs 2

In [ ]:
# Fig. 7: lambda_h of the Dynamic scheduler
!python scripts/run_suite.py --suite lambda_mnist --preset paper --jobs 2

In [ ]:
# 4) tables + figures, then zip everything for download
!python scripts/plot.py --results results
!zip -qr /kaggle/working/results.zip results -x 'results/ckpt/*'
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*accuracy.png')): display(Image(f))